In [2]:
# Core RAG Frameworks
!pip install langchain
!pip install langchain-openai
!pip install langchain-community
!pip install langchain-text-splitters
!pip install langchain-experimental
!pip install langgraph


In [3]:
import json
import os

from openai import OpenAI
from google.colab import userdata

from langchain_core.documents import Document
from langchain_experimental.text_splitter import SemanticChunker  # AI-powered semantic chunking
from langchain_community.vectorstores import Chroma  # Vector database
from langchain_openai import OpenAIEmbeddings  # OpenAI embedding function
from langchain_core.documents import Document  # Document abstraction


from langchain_text_splitters import (  # Various splitting strategies
    RecursiveCharacterTextSplitter,  # Best general-purpose splitter
    CharacterTextSplitter,  # Simple split by character count
    MarkdownHeaderTextSplitter,  # Splits based on markdown headers
    HTMLHeaderTextSplitter  # Splits based on HTML tags
)


/tmp/ipykernel_5087/2552726703.py:8: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker  # AI-powered semantic chunking


In [4]:
client = OpenAI(api_key=userdata.get('OPENAI_APIKEY'))

In [5]:
OPENAI_EMBEDDING_MODEL='text-embedding-3-small'
# embedding_model = os.getenv('OPENAI_EMBEDDING_MODEL', 'text-embedding-3-small')
embedding_model = OPENAI_EMBEDDING_MODEL
embedding_dim = 1536  # Number of dimensions in the embedding vector
print(f"Using OpenAI model: {embedding_model}")
print(f"Embedding dimension: {embedding_dim}")


Using OpenAI model: text-embedding-3-small
Embedding dimension: 1536


In [6]:
# =============================================================================
# LOAD DATA: Support Tickets
# =============================================================================
print("\nLoading support tickets...")
with open('./sample_data/synthetic_tickets.json', 'r') as f:
    tickets = json.load(f)
print(f"Loaded {len(tickets)} support tickets")


Loading support tickets...
Loaded 3 support tickets


In [7]:
# Display sample ticket - understand what we're working with
print("\n" + "="*80)
print("SAMPLE TICKET:")
print("="*80)
sample = tickets[0]
print(f"ID: {sample['ticket_id']}")
print(f"Title: {sample['title']}")
print(f"Description: {sample['description'][:200]}...")
print(f"Category: {sample['category']}")
print(f"Priority: {sample['priority']}")

print("\n" + "="*80)
print("PART 1: Chunking Strategies")
print("="*80)


SAMPLE TICKET:
ID: TIC-1
Title: Cannot reset password
Description: I tried resetting my password but never received the email....
Category: Authentication
Priority: High

PART 1: Chunking Strategies


In [8]:
# -----------------------------------------------------------------------------
# First, convert our ticket data into LangChain Document objects
# Document = page_content (text) + metadata (structured info for filtering)
# -----------------------------------------------------------------------------

documents = []
for ticket in tickets:
    # Combine all ticket fields into a single text block
    # TIP: Include all relevant context that helps understand the document
    full_text = f"""
Ticket ID: {ticket['ticket_id']}
Title: {ticket['title']}
Category: {ticket['category']}
Priority: {ticket['priority']}
Description: {ticket['description']}
Resolution: {ticket['resolution']}
    """.strip()


    # Create Document object with metadata
    # Metadata is CRUCIAL - it enables filtering later!
    # Example: "Find similar tickets, but only in the 'Authentication' category"
    doc = Document(
        page_content=full_text,  # The actual text content
        metadata={
            'ticket_id': ticket['ticket_id'],   # For identifying results
            'category': ticket['category'],      # For category filtering
            'priority': ticket['priority']       # For priority filtering
        }
    )
    documents.append(doc)

print(f"Created {len(documents)} documents")
print(f"\nSample document length: {len(documents[0].page_content)} characters")

Created 3 documents

Sample document length: 235 characters


In [9]:
# =============================================================================
# STRATEGY 1: Fixed-Size Chunking
# =============================================================================

print("\n--- Strategy 1: Fixed-Size Chunking ---")

fixed_splitter = CharacterTextSplitter(
    chunk_size=200,      # Maximum characters per chunk
    chunk_overlap=20,    # Characters to repeat between chunks (10% overlap)
    separator="\n"       # Prefer splitting on newlines when possible
)
fixed_chunks = fixed_splitter.split_documents(documents)

print(f"✓ Created {len(fixed_chunks)} chunks")
print(f"  Chunk size: 200 chars, Overlap: 20 chars")
print(f"  Sample chunk: {fixed_chunks[0].page_content[:100]}...")


--- Strategy 1: Fixed-Size Chunking ---
✓ Created 6 chunks
  Chunk size: 200 chars, Overlap: 20 chars
  Sample chunk: Ticket ID: TIC-1
Title: Cannot reset password
Category: Authentication
Priority: High
Description: I...


In [10]:
# =============================================================================
# STRATEGY 2: Recursive Character Splitting (RECOMMENDED DEFAULT)
# =============================================================================

print("\n--- Strategy 2: Recursive Character Splitting ---")

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,      # Max characters per chunk
    chunk_overlap=50,    # 50 char overlap (~17%)
    # Separators tried in ORDER - most specific first!
    separators=[
        "\n\n",  # 1st: Paragraph breaks (best split point)
        "\n",    # 2nd: Line breaks
        ". ",    # 3rd: Sentence boundaries
        " "      # 4th: Word boundaries (last resort for text)
    ]
)
recursive_chunks = recursive_splitter.split_documents(documents)

print(f"✓ Created {len(recursive_chunks)} chunks")
print(f"  Tries to split on paragraph/sentence boundaries")
print(f"  Sample chunk: {recursive_chunks[0].page_content[:100]}...")



--- Strategy 2: Recursive Character Splitting ---
✓ Created 3 chunks
  Tries to split on paragraph/sentence boundaries
  Sample chunk: Ticket ID: TIC-1
Title: Cannot reset password
Category: Authentication
Priority: High
Description: I...


In [11]:
# =============================================================================
# STRATEGY 3: Semantic Chunking (Embedding-Based)
# =============================================================================

print("\n--- Strategy 3: Semantic Chunking ---")
print("  Note: Semantic chunking uses embeddings to find natural break points")

# Initialize OpenAI embeddings for semantic chunker
# IMPORTANT: This costs money! Each sentence needs an embedding API call
embeddings_model = OpenAIEmbeddings(
    model='text-embedding-3-small',
    api_key=userdata.get('OPENAI_APIKEY')  # Explicitly passing the API key
)

# Demo with a paragraph that has CLEAR topic shifts
# Using completely unrelated domains for clearer separation
demo_paragraph = """
The Mars rover collected soil samples from the Jezero crater last week. Scientists believe these rocks may contain signs of ancient microbial life. NASA plans to retrieve these samples in a future mission. The discovery could reshape our understanding of life in the solar system.

Grandma's apple pie recipe starts with peeling six large Granny Smith apples. Mix flour, sugar, and cinnamon for the filling. Roll the dough thin and crimp the edges carefully. Bake at 375 degrees for 45 minutes until golden brown.

The defendant was charged with breach of contract under Section 12. The plaintiff seeks damages of fifty thousand dollars plus legal fees. Both parties agreed to mediation before proceeding to trial. The judge scheduled the preliminary hearing for next month.
"""

print("\n   Demo Text (3 distinct topics):")
print("  " + "-"*70)
print("  Topic 1: Space exploration (sentences 1-4)")
print("  Topic 2: Cooking recipe (sentences 5-8)")
print("  Topic 3: Legal case (sentences 9-12)")
print("  " + "-"*70)

semantic_splitter = SemanticChunker(
    embeddings=embeddings_model,
    # How to detect "topic change":
    # - "percentile": Split where similarity is in bottom X percentile
    # - "standard_deviation": Split where similarity is X std devs below mean
    # - "interquartile": Split where similarity is below Q1 - 1.5*IQR (outlier detection)
    breakpoint_threshold_type="standard_deviation",
    breakpoint_threshold_amount=1.0  # Split when similarity drops 1 std dev below mean
)

demo_doc = Document(page_content=demo_paragraph.strip())
semantic_chunks = semantic_splitter.split_documents([demo_doc])
print(f"\n✓ Created {len(semantic_chunks)} chunks")
print("  Note: Semantic chunking results vary based on embedding model and threshold settings")

# Show each semantic chunk
print("\n  Resulting Semantic Chunks:")
print("  " + "-"*70)
for i, chunk in enumerate(semantic_chunks):
    print(f"\n  Chunk {i+1} ({len(chunk.page_content)} chars):")
    print("  " + "~"*60)
    # Show full content for clarity
    for line in chunk.page_content.strip().split('\n'):
        if line.strip():
            print(f"    {line.strip()}")
    print("  " + "~"*60)

print("\n  ✨ The chunker attempts to detect topic shifts between space → cooking → legal")
print("  Adjust breakpoint_threshold_amount (lower = more sensitive) if results vary.")


--- Strategy 3: Semantic Chunking ---
  Note: Semantic chunking uses embeddings to find natural break points

   Demo Text (3 distinct topics):
  ----------------------------------------------------------------------
  Topic 1: Space exploration (sentences 1-4)
  Topic 2: Cooking recipe (sentences 5-8)
  Topic 3: Legal case (sentences 9-12)
  ----------------------------------------------------------------------

✓ Created 3 chunks
  Note: Semantic chunking results vary based on embedding model and threshold settings

  Resulting Semantic Chunks:
  ----------------------------------------------------------------------

  Chunk 1 (205 chars):
  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    The Mars rover collected soil samples from the Jezero crater last week. Scientists believe these rocks may contain signs of ancient microbial life. NASA plans to retrieve these samples in a future mission.
  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

  Chunk 2 (2

In [12]:
# =============================================================================
# STRATEGY 4: Markdown Structure-Aware Splitting
# =============================================================================

print("\n--- Strategy 4: Markdown Header Splitting ---")

# Sample markdown documentation (simulating a knowledge base article)
markdown_doc = """
# Database Troubleshooting Guide

## Connection Issues

### Timeout Errors
If you encounter timeout errors, check the connection string and ensure the database server is reachable.
Increase the connection timeout value in your configuration.

### Authentication Failures
Verify your credentials are correct. Check for expired passwords or locked accounts.
Ensure the user has proper permissions on the database.

## Performance Problems

### Slow Queries
Analyze query execution plans using EXPLAIN.
Consider adding indexes on frequently queried columns.
Review and optimize JOIN operations.

### High CPU Usage
Monitor long-running queries.
Check for missing indexes causing table scans.
"""

# Define which headers to split on
# Format: (header_marker, metadata_key)
headers_to_split_on = [
    ("#", "Header 1"),    # H1 tags
    ("##", "Header 2"),   # H2 tags
    ("###", "Header 3"),  # H3 tags
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False  # Keep headers in the chunk content (usually want True)
)
md_chunks = markdown_splitter.split_text(markdown_doc)

print(f"✓ Created {len(md_chunks)} chunks from markdown")
print(f"  Preserves document structure and header context")

print("\n--- Strategy 4: Markdown Header Splitting ---")

# Sample markdown documentation (simulating a knowledge base article)
markdown_doc = """
# Database Troubleshooting Guide

## Connection Issues

### Timeout Errors
If you encounter timeout errors, check the connection string and ensure the database server is reachable.
Increase the connection timeout value in your configuration.

### Authentication Failures
Verify your credentials are correct. Check for expired passwords or locked accounts.
Ensure the user has proper permissions on the database.

## Performance Problems

### Slow Queries
Analyze query execution plans using EXPLAIN.
Consider adding indexes on frequently queried columns.
Review and optimize JOIN operations.

### High CPU Usage
Monitor long-running queries.
Check for missing indexes causing table scans.
"""

# Define which headers to split on
# Format: (header_marker, metadata_key)
headers_to_split_on = [
    ("#", "Header 1"),    # H1 tags
    ("##", "Header 2"),   # H2 tags
    ("###", "Header 3"),  # H3 tags
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False  # Keep headers in the chunk content (usually want True)
)
md_chunks = markdown_splitter.split_text(markdown_doc)

print(f"✓ Created {len(md_chunks)} chunks from markdown")
print(f"  Preserves document structure and header context")
print("  " + "="*60)

for i, chunk in enumerate(md_chunks):
    print(f"\n  Chunk {i+1} ({len(chunk.page_content)} chars):")
    print("  " + "~"*60)
    # Show full content for clarity
    for line in chunk.page_content.strip().split('\n'):
        if line.strip():
            print(f"    {line.strip()}")


print("  " + "="*60)




--- Strategy 4: Markdown Header Splitting ---
✓ Created 4 chunks from markdown
  Preserves document structure and header context

--- Strategy 4: Markdown Header Splitting ---
✓ Created 4 chunks from markdown
  Preserves document structure and header context

  Chunk 1 (243 chars):
  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Database Troubleshooting Guide
    ## Connection Issues
    ### Timeout Errors
    If you encounter timeout errors, check the connection string and ensure the database server is reachable.
    Increase the connection timeout value in your configuration.

  Chunk 2 (168 chars):
  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    ### Authentication Failures
    Verify your credentials are correct. Check for expired passwords or locked accounts.
    Ensure the user has proper permissions on the database.

  Chunk 3 (179 chars):
  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    ## Performance Problems
    ### Sl

In [13]:
# =============================================================================
# STRATEGY 5: HTML Structure-Aware Splitting
# =============================================================================

print("\n--- Strategy 5: HTML Header Splitting ---")

# Sample HTML documentation (simulating a scraped help page)
html_doc = """
<!DOCTYPE html>
<html>
<body>
    <h1>Email Configuration Guide</h1>

    <h2>SMTP Settings</h2>
    <p>Configure your SMTP server settings in the admin panel. Use port 587 for TLS or port 465 for SSL.</p>

    <h3>Common SMTP Servers</h3>
    <p>Gmail: smtp.gmail.com, Outlook: smtp.office365.com, Yahoo: smtp.mail.yahoo.com</p>

    <h2>IMAP Configuration</h2>
    <p>Set up IMAP to sync your emails across devices. Use port 993 for secure connections.</p>

    <h3>Folder Mapping</h3>
    <p>Map your email folders to the appropriate IMAP folders for proper synchronization.</p>
</body>
</html>
"""

# Map HTML tags to metadata keys
headers_to_split_on_html = [
    ("h1", "Header 1"),
    ("h2", "Header 2"),
    ("h3", "Header 3"),
]

html_splitter = HTMLHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on_html
)
html_chunks = html_splitter.split_text(html_doc)

print(f"✓ Created {len(html_chunks)} chunks from HTML")
print(f"  Respects HTML semantic structure")

print("  " + "="*60)

for i, chunk in enumerate(html_chunks):
    print(f"\n  Chunk {i+1} ({len(chunk.page_content)} chars):")
    print("  " + "~"*60)
    # Show full content for clarity
    for line in chunk.page_content.strip().split('\n'):
        if line.strip():
            print(f"    {line.strip()}")


print("  " + "="*60)





--- Strategy 5: HTML Header Splitting ---
✓ Created 9 chunks from HTML
  Respects HTML semantic structure

  Chunk 1 (25 chars):
  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    Email Configuration Guide

  Chunk 2 (13 chars):
  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    SMTP Settings

  Chunk 3 (97 chars):
  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    Configure your SMTP server settings in the admin panel. Use port 587 for TLS or port 465 for SSL.

  Chunk 4 (19 chars):
  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    Common SMTP Servers

  Chunk 5 (78 chars):
  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    Gmail: smtp.gmail.com, Outlook: smtp.office365.com, Yahoo: smtp.mail.yahoo.com

  Chunk 6 (18 chars):
  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    IMAP Configuration

  Chunk 7 (84 chars):
  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    Set up 

In [14]:
# Vector Stores
!pip install chromadb
!pip install langchain-chroma

In [15]:
# =============================================================================
# STRATEGY 6: No Chunking (Whole Documents)
# =============================================================================

print("\n--- Strategy 6: Whole Documents (No Chunking) ---")
print(f"✓ Using {len(documents)} whole documents")
print(f"  Good for small documents like our tickets")

print("\n" + "="*80)
print("PART 2: Chroma Vector Store")
print("="*80)

# Use LangChain's embedding wrapper (handles API calls internally)
embeddings_model = OpenAIEmbeddings(
    model='text-embedding-3-small',
    api_key=userdata.get('OPENAI_APIKEY')
)


print("\nBuilding Chroma vector store...")
existing_store = Chroma(
    collection_name="support_tickets",
    persist_directory="./chroma_db"
)
existing_store.delete_collection()

chroma_store = Chroma.from_documents(
    documents=documents,              # Our LangChain Document objects
    embedding=embeddings_model,       # OpenAI embeddings
    collection_name="support_tickets",# Like a \"table\" in a database
    persist_directory="./chroma_db"   # Save to disk for persistence
)
print("✓ Chroma store created and persisted")


--- Strategy 6: Whole Documents (No Chunking) ---
✓ Using 3 whole documents
  Good for small documents like our tickets

PART 2: Chroma Vector Store

Building Chroma vector store...


/tmp/ipykernel_5087/2207256256.py:21: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  existing_store = Chroma(


✓ Chroma store created and persisted


In [16]:
# -----------------------------------------------------------------------------
# Basic Similarity Search
# -----------------------------------------------------------------------------
query = "Authentication problems after password reset"

print(f"\nSearching in Chroma: '{query}'")
chroma_results = chroma_store.similarity_search(query, k=3)

print(f"\nTop {len(chroma_results)} results:")
for i, doc in enumerate(chroma_results, 1):
    print(f"\n#{i}")
    print(f"Ticket: {doc.metadata['ticket_id']}")
    print(f"Category: {doc.metadata['category']}")


Searching in Chroma: 'Authentication problems after password reset'

Top 3 results:

#1
Ticket: TIC-1
Category: Authentication

#2
Ticket: TIC-3
Category: Networking

#3
Ticket: TIC-2
Category: Performance


In [17]:
# -----------------------------------------------------------------------------
# MMR Search (Maximal Marginal Relevance)
# -----------------------------------------------------------------------------

print("\n--- Using MMR for Diverse Results ---")
mmr_results = chroma_store.max_marginal_relevance_search(query, k=3)

print(f"\nMMR Results (more diverse):")
for i, doc in enumerate(mmr_results, 1):
    print(f"\n#{i}")
    print(f"Ticket: {doc.metadata['ticket_id']}")
    print(f"Title: {tickets[int(doc.metadata['ticket_id'].split('-')[1]) - 1]['title']}")


--- Using MMR for Diverse Results ---

MMR Results (more diverse):

#1
Ticket: TIC-1
Title: Cannot reset password

#2
Ticket: TIC-3
Title: VPN connection dropping

#3
Ticket: TIC-2
Title: Slow dashboard loading
